In [3]:
from pathlib import Path
import json
from dataset import device_for_run, load_store
from gpt2 import load_gpt
from reward_model import load_reward
from evaluate import evaluate_variant
from analysis import plot_comparison, plot_rerank_tradeoff, plot_training_diagnostics, compare_reward_hacking
from tracking import start_task


In [4]:
ROOT = Path.cwd() if (Path.cwd() / 'rl.py').exists() else Path.cwd() / 'task2'
BASE = ROOT / 'checkpoints' / 'gpt2'
ANALYSIS_NAME = 'final_test_h333'
RL_VARIANT = 'h3_relevance_03_v2'
RUN_TEST = True
SELECTED_RERANK_ALPHA = 0.3


RESULTS = ROOT / 'results' / 'analysis_v2' / ANALYSIS_NAME
if RESULTS.exists():
    raise FileExistsError(f'Choose a new ANALYSIS_NAME: {RESULTS}')
RESULTS.mkdir(parents=True)
LOG_CLEARML = True

RERANK_N = 100
RERANK_ALPHAS = [0.0, 0.1, 0.3, 0.5, 1.0]



device = device_for_run()
gpt_settings = json.loads((BASE / 'settings.json').read_text())
store = load_store(ROOT / 'data' / 'Yambda', max_length=gpt_settings['max_length'], vocab_path=BASE / 'vocab.json')
base = load_gpt(BASE, device).eval()
reward42, reward_settings = load_reward(ROOT / 'checkpoints' / 'reward_seed_42', store.catalog.vocab_size, device)
reward44, _ = load_reward(ROOT / 'checkpoints' / 'reward_seed_44', store.catalog.vocab_size, device)


Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

In [5]:
comparisons = {}
plans = [('gpt2', base, 0, None)]
for alpha in RERANK_ALPHAS:
    plans.append((f'rerank_alpha_{alpha:g}', base, RERANK_N, alpha))
plans.append(('rerank_reward_only', base, RERANK_N, None))
checkpoint = ROOT / 'checkpoints' / RL_VARIANT
if checkpoint.exists():
    plans.append((RL_VARIANT, load_gpt(checkpoint, device).eval(), 0, None))
for name, policy, rerank_n, alpha in plans:
    task = start_task(f'validation {name}', {'variant': name, 'rerank_n': rerank_n, 'alpha': alpha}, LOG_CLEARML)
    comparisons[name] = evaluate_variant(store, policy, base, reward42, reward44,
                                         RESULTS / f'{name}_validation.json', device,
                                         rerank_n=rerank_n, rerank_alpha=alpha,
                                         reward_function='ratio', like_weight=reward_settings.like_weight,
                                         task=task, include_test=RUN_TEST)
    if task:
        task.close()
    print(name, {key: comparisons[name]['validation']['summary'][key + '_mean']
                 for key in ('ndcg@20', 'recall@20', 'predicted_reward@20', 'independent_reward@20')})


Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

ClearML Task: created new task id=e1309aa348744a5f8dc86f6ee602543b
2026-10-25 21:07:05,504 - clearml.Repository Detection - WARNING - Can't get url information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
2026-10-25 21:07:05,539 - clearml.Repository Detection - WARNING - Can't get branch information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
2026-10-25 21:07:05,558 - clearml.Repository Detection - WARNING - Can't get commit information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
2026-10-25 21:07:05,630 - clearml.Repository Detection - WARNING - Can't get diff information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/e1309aa348744a5f8dc86f6ee602543b/output/log
2026-10-25 21:07:05,969 - clearml.resource_monitor - WARNING - Could not fetch GPU stat

Can't get url information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get branch information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get commit information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get diff information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages


ClearML Task: created new task id=3a0b95e8012c4b1b9db13960e641f54b


Could not fetch GPU stats: NVML Shared Library Not Found


...

ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/ff73fbe5cbb74d30a25a2262393a48c0/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
rerank_reward_only {'ndcg@20': 0.021801236448043708, 'recall@20': 0.05952380952380952, 'predicted_reward@20': 0.7162370085716248, 'independent_reward@20': 0.642655611038208}


Can't get url information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get branch information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get commit information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get diff information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages


ClearML Task: created new task id=f3938e2f4d604de0a639647d1564314b


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/f3938e2f4d604de0a639647d1564314b/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
h3_relevance_03_v2 {'ndcg@20': 0.08484591335724442, 'recall@20': 0.15079365079365079, 'predicted_reward@20': 0.6330159306526184, 'independent_reward@20': 0.6384996771812439}


In [6]:
plot_comparison(comparisons, RESULTS / 'validation_comparison.png', split='validation')
plot_rerank_tradeoff(comparisons, RESULTS / 'rerank_tradeoff.png')
display(compare_reward_hacking(comparisons, split='validation'))
if checkpoint.exists():
    history = json.loads((checkpoint / 'history.json').read_text())
    plot_training_diagnostics(history, RESULTS / f'{RL_VARIANT}_training.png')


,model,reward_function,trained_reward@20,independent_reward@20,reward_gap,recall@20,kl_vs_base
0,gpt2,ratio,0.631078,0.637662,-0.006583,0.146825,0.000000
1,rerank_alpha_0,ratio,0.631078,0.637662,-0.006583,0.146825,0.000000
2,rerank_alpha_0.1,ratio,0.637176,0.637807,-0.000632,0.146825,0.000000
3,rerank_alpha_0.3,ratio,0.649475,0.638765,0.010710,0.150794,0.000000
4,rerank_alpha_0.5,ratio,0.660957,0.639561,0.021396,0.154762,0.000000
5,rerank_alpha_1,ratio,0.685511,0.640933,0.044578,0.126984,0.000000
6,rerank_reward_only,ratio,0.716237,0.642656,0.073581,0.059524,0.000000
7,h3_relevance_03_v2,ratio,0.633016,0.638500,-0.005484,0.150794,0.027098


In [7]:
from analysis import reward_bias_diagnostics
RUN_REWARD_DIAGNOSTICS = False
if RUN_REWARD_DIAGNOSTICS:
    diagnostic = reward_bias_diagnostics(store, base, reward42, reward44,
                                         ROOT / 'results' / 'reward_events.parquet', device)
    (RESULTS / 'reward_bias_diagnostics.json').write_text(json.dumps(diagnostic, indent=2))
    print(diagnostic)


In [8]:
if RUN_TEST:
    selected_names = ['gpt2', RL_VARIANT]
    if SELECTED_RERANK_ALPHA is not None:
        selected_names.append(f'rerank_alpha_{SELECTED_RERANK_ALPHA:g}')
    for name in selected_names:
        policy = load_gpt(ROOT / 'checkpoints' / name, device).eval() if name == RL_VARIANT else base
        rerank_n = RERANK_N if name.startswith('rerank_alpha_') else 0
        alpha = SELECTED_RERANK_ALPHA if rerank_n else None
        task = start_task(f'test {name}', {'variant': name}, LOG_CLEARML)
        result = evaluate_variant(store, policy, base, reward42, reward44,
                                  RESULTS / f'{name}_test.json', device,
                                  rerank_n=rerank_n, rerank_alpha=alpha,
                                  reward_function='ratio', like_weight=reward_settings.like_weight,
                                  task=task, include_test=True)
        if task:
            task.close()
        print(name, result['test']['summary'])


Can't get url information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get branch information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get commit information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get diff information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages


ClearML Task: created new task id=4686555778a24dafa7121c5eb4548b9a


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/4686555778a24dafa7121c5eb4548b9a/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
gpt2 {'queries_mean': 283.0, 'queries_std': 0.0, 'target_outside_catalog_mean': 71.8, 'target_outside_catalog_std': 5.769652406245015, 'target_previously_seen_mean': 140.6, 'target_previously_seen_std': 7.676804890925217, 'eligible_queries_mean': 211.2, 'eligible_queries_std': 5.769652406245014, 'catalog_items_mean': 30000.0, 'catalog_items_std': 0.0, 'recall@5_mean': 0.07102473498233217, 'recall@5_std': 0.012735018064334995, 'ndcg@5_mean': 0.052656888871390625, 'ndcg@5_std': 0.008674991016580737, 'eligible_recall@5_mean': 0.09524448511134148, 'eligible_recall@5_std': 0.017386332059857935, 'eligible_ndcg@5_mean': 0.070621376304968, 'eligible_ndcg@5_std': 0.011903668612010665, 'coverage@5_mean': 0.03959, 'coverage@5_std': 0.00038266447282891573, 'popular_fraction@5_m

Loading weights:   0%|          | 0/28 [00:00<?, ?it/s]

...

ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/fce46ef3561549f8bc243fb034ba1024/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
h3_relevance_03_v2 {'queries_mean': 283.0, 'queries_std': 0.0, 'target_outside_catalog_mean': 71.8, 'target_outside_catalog_std': 5.769652406245015, 'target_previously_seen_mean': 140.6, 'target_previously_seen_std': 7.676804890925217, 'eligible_queries_mean': 211.2, 'eligible_queries_std': 5.769652406245014, 'catalog_items_mean': 30000.0, 'catalog_items_std': 0.0, 'recall@5_mean': 0.06996466431095408, 'recall@5_std': 0.010375837712859949, 'ndcg@5_mean': 0.0518974446969422, 'ndcg@5_std': 0.007918263884244686, 'eligible_recall@5_mean': 0.09378390465897528, 'eligible_recall@5_std': 0.013920564576604894, 'eligible_ndcg@5_mean': 0.0695807682943527, 'eligible_ndcg@5_std': 0.010680316079117047, 'coverage@5_mean': 0.037803333333333335, 'coverage@5_std': 0.00032373209469529

Can't get url information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get branch information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get commit information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages
Can't get diff information for git repo in /Users/dshhrv/tlab recsys/task2/.venv/lib/python3.11/site-packages


ClearML Task: created new task id=a3f84501495a4a0a82399b27d10ab817


Could not fetch GPU stats: NVML Shared Library Not Found


ClearML results page: https://app.clear.ml/projects/b41b8f398f364b9cbe1b273c663b6cea/tasks/a3f84501495a4a0a82399b27d10ab817/output/log
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
rerank_alpha_0.3 {'queries_mean': 283.0, 'queries_std': 0.0, 'target_outside_catalog_mean': 71.8, 'target_outside_catalog_std': 5.769652406245015, 'target_previously_seen_mean': 140.6, 'target_previously_seen_std': 7.676804890925217, 'eligible_queries_mean': 211.2, 'eligible_queries_std': 5.769652406245014, 'catalog_items_mean': 30000.0, 'catalog_items_std': 0.0, 'recall@5_mean': 0.06925795053003533, 'recall@5_std': 0.013346728362040327, 'ndcg@5_mean': 0.05251352572976777, 'ndcg@5_std': 0.009573363896153682, 'eligible_recall@5_mean': 0.09283451237916518, 'eligible_recall@5_std': 0.017780819526970016, 'eligible_ndcg@5_mean': 0.07039723456528767, 'eligible_ndcg@5_std': 0.012805109879786333, 'coverage@5_mean': 0.039246666666666666, 'coverage@5_std': 0.00030151878921041

In [9]:
import pandas as pd

rows = []

for name, result in comparisons.items():
    summary = result["validation"]["summary"]
    rows.append({
        "Model": name,
        "Recall@20": summary["recall@20_mean"],
        "NDCG@20": summary["ndcg@20_mean"],
        "Reward@20": summary["independent_reward@20_mean"],
        "Coverage@20": summary["coverage@20_mean"],
    })

df = pd.DataFrame(rows).set_index("Model")
display(df.round(5))

,Recall@20,NDCG@20,Reward@20,Coverage@20
Model,,,,
gpt2,0.14683,0.08450,0.63766,0.11697
rerank_alpha_0,0.14683,0.08450,0.63766,0.11697
rerank_alpha_0.1,0.14683,0.08355,0.63781,0.11653
rerank_alpha_0.3,0.15079,0.08338,0.63877,0.11533
rerank_alpha_0.5,0.15476,0.08002,0.63956,0.11440
rerank_alpha_1,0.12698,0.06574,0.64093,0.10877
rerank_reward_only,0.05952,0.02180,0.64266,0.09783
h3_relevance_03_v2,0.15079,0.08485,0.63850,0.11093


In [10]:
import pandas as pd

rows = []

for name in ['gpt2', 'rerank_alpha_0.3', 'h3_relevance_03_v2']:
    result = comparisons[name]

    if 'test' not in result:
        print(f'{name}: test еще не посчитан')
        continue

    s = result['test']['summary']

    rows.append({
        'Model': name,
        'Recall@20': f"{s['recall@20_mean']:.4f} ± {s['recall@20_std']:.4f}",
        'NDCG@20': f"{s['ndcg@20_mean']:.4f} ± {s['ndcg@20_std']:.4f}",
        'Reward@20': f"{s['independent_reward@20_mean']:.4f} ± {s['independent_reward@20_std']:.4f}",
        'Coverage@20': f"{s['coverage@20_mean']:.4f} ± {s['coverage@20_std']:.4f}"
    })

if rows:
    display(pd.DataFrame(rows).set_index('Model'))

,Recall@20,NDCG@20,Reward@20,Coverage@20
Model,,,,
gpt2,0.1163 ± 0.0153,0.0653 ± 0.0085,0.6355 ± 0.0026,0.1250 ± 0.0015
rerank_alpha_0.3,0.1141 ± 0.0162,0.0651 ± 0.0099,0.6364 ± 0.0026,0.1239 ± 0.0015
h3_relevance_03_v2,0.1216 ± 0.0186,0.0660 ± 0.0093,0.6364 ± 0.0025,0.1182 ± 0.0017
